# City-Wide Inference Mosaic & km² Quantification

In [1]:
import csv
import os
import random
import time
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
try:
    import timm
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "-q", "timm"])
    import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from skimage.filters import threshold_otsu
from sklearn.metrics import cohen_kappa_score
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore", category=UserWarning, module="timm.layers.interpolate")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def resolve_path(candidates, default):
    for c in candidates:
        if c and os.path.exists(c):
            return c
    return default

DATA_DIR = resolve_path([os.getenv("DATA_DIR", ""), "./data", "./dataset", "/kaggle/input/lulc-chennai", "../data"], "./data")
OUT_DIR = os.getenv("OUT_DIR", "./outputs")
os.makedirs(OUT_DIR, exist_ok=True)
NUM_CLASSES = 8
CKPT_NAME = "best_model_final_5.pth"
CVA_WATER_NDWI_THRESHOLD = 0.35
DISPLAY_CONFIDENCE_TAU = 0.60

CONFIG = {
    "img_size": 128,
    "in_chans": 9,
    "num_classes": NUM_CLASSES,
    "batch_size": 32,
    "num_workers": 4,
    "lr": 5.0e-5,
    "weight_decay": 6.358359e-4,
    "focal_gamma": 0.6136,
    "backbone_lr_ratio": 0.2719,
    "epochs": 160,
    "freeze_epochs": 15,
    "early_stop_patience": 20,
    "grad_clip_norm": 1.0,
    "label_smoothing": 0.1,
    "class_weight_cap": 5.0,
    "change_pos_weight_cap": 8.0,
    "change_bce_weight": 0.70,
    "change_dice_weight": 0.30,
    "change_task_weight": 1.0,
    "backbone": "swin_tiny_patch4_window7_224",
    "attn_heads": 8,
    "cross_attn_heads": 8,
    "num_layers": 3,
}

print(f"Device: {device}")

Device: cuda


## Data Loading

In [2]:
patches_2017 = np.load(os.path.join(DATA_DIR, "patches_2017.npy"))
patches_2019 = np.load(os.path.join(DATA_DIR, "patches_2019.npy"))
patches_2024 = np.load(os.path.join(DATA_DIR, "patches_2024.npy"))
patches_2025 = np.load(os.path.join(DATA_DIR, "patches_2025.npy"))
labels = np.load(os.path.join(DATA_DIR, "label_patches.npy"))

YEAR_ARRAYS = {
    2017: patches_2017,
    2019: patches_2019,
    2024: patches_2024,
    2025: patches_2025,
}
LABEL_FOR_YEAR = {year: labels for year in YEAR_ARRAYS}

n_patches = labels.shape[0]
rng = np.random.RandomState(SEED)
shuffled_idx = rng.permutation(n_patches)
n_train = int(n_patches * 0.80)
train_patch_idx = shuffled_idx[:n_train]
val_patch_idx = shuffled_idx[n_train:]
print(f"Split: {len(train_patch_idx)} train / {len(val_patch_idx)} val")


Split: 2774 train / 694 val


## Spectral Processing

In [3]:
def to_nine_band(patch_5band, eps=1e-6):
    b2, b3, b4, b8, b11 = patch_5band[:5]
    ndvi = (b8 - b4) / (b8 + b4 + eps)
    ndwi = (b3 - b8) / (b3 + b8 + eps)
    ndbi = (b11 - b8) / (b11 + b8 + eps)
    mndwi = (b3 - b11) / (b3 + b11 + eps)
    return np.stack([b2, b3, b4, b8, b11, ndvi, ndwi, ndbi, mndwi], axis=0).astype(np.float32)


def normalize_bands(bands):
    out = bands.copy()
    out[:5] = np.clip(out[:5], 0, 1)
    out[5:] = (np.clip(out[5:], -1, 1) + 1) / 2
    return out

## Model Architecture

In [4]:
class UNetDecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(out_ch+skip_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),         nn.BatchNorm2d(out_ch), nn.ReLU(True))
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

class ASPP(nn.Module):
    def __init__(self, in_ch, out_ch, rates=[1,2,4]):
        super().__init__()
        self.stages = nn.ModuleList([nn.Sequential(
            nn.Conv2d(in_ch,out_ch,1,bias=False),nn.BatchNorm2d(out_ch),nn.ReLU(True))])
        for r in rates:
            self.stages.append(nn.Sequential(
                nn.Conv2d(in_ch,out_ch,3,padding=r,dilation=r,bias=False),nn.BatchNorm2d(out_ch),nn.ReLU(True)))
        self.pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),nn.Conv2d(in_ch,out_ch,1,bias=False),nn.BatchNorm2d(out_ch),nn.ReLU(True))
        self.project = nn.Sequential(
            nn.Conv2d(out_ch*(len(rates)+2),out_ch,1,bias=False),nn.BatchNorm2d(out_ch),nn.ReLU(True))
    def forward(self, x):
        feats = [s(x) for s in self.stages]
        p = F.interpolate(self.pool(x), size=x.shape[-2:], mode="bilinear", align_corners=False)
        feats.append(p)
        return self.project(torch.cat(feats, dim=1))

def widen_patch_embed(backbone, in_chans):
    proj = backbone.patch_embed.proj; old_w = proj.weight.data
    new_proj = nn.Conv2d(in_chans, proj.out_channels, proj.kernel_size, proj.stride, proj.padding,
                         bias=proj.bias is not None)
    with torch.no_grad():
        new_proj.weight[:,:3] = old_w
        if in_chans > 3:
            new_proj.weight[:,3:] = old_w.mean(1,keepdim=True).repeat(1,in_chans-3,1,1)
        if proj.bias is not None: new_proj.bias[:] = proj.bias.data
    backbone.patch_embed.proj = new_proj

def to_nchw(feat, chs):
    if feat.dim()==4 and feat.shape[1]!=feat.shape[-1] and feat.shape[-1] in chs:
        feat = feat.permute(0,3,1,2).contiguous()
    return feat

class BSALayer(nn.Module):
    def __init__(self, dim, heads=8):
        super().__init__()
        self.n1 = nn.LayerNorm(dim); self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.n2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim,dim*4),nn.GELU(),nn.Linear(dim*4,dim))
    def forward(self, x):
        h=self.n1(x); x=x+self.attn(h,h,h,need_weights=False)[0]
        h=self.n2(x); return x+self.mlp(h)

class BottleneckSelfAttention(nn.Module):
    def __init__(self, dim, heads=8, num_layers=1):
        super().__init__()
        self.layers = nn.ModuleList([BSALayer(dim,heads) for _ in range(num_layers)])
        self.norm   = nn.LayerNorm(dim)
    def forward(self, x):
        B,C,H,W = x.shape
        t = x.permute(0,2,3,1).reshape(B,H*W,C)
        for l in self.layers: t=l(t)
        return self.norm(t).reshape(B,H,W,C).permute(0,3,1,2)

class CrossTransformerBlock(nn.Module):
    def __init__(self, dim, heads=8):
        super().__init__()
        self.na=nn.LayerNorm(dim); self.nb=nn.LayerNorm(dim)
        self.attn12=nn.MultiheadAttention(dim,heads,batch_first=True)
        self.attn21=nn.MultiheadAttention(dim,heads,batch_first=True)
        self.nc=nn.LayerNorm(dim); self.nd=nn.LayerNorm(dim)
        self.proj=nn.Linear(dim*2,dim)
    def forward(self, f1, f2):
        B,C,H,W = f1.shape
        t1=f1.permute(0,2,3,1).reshape(B,H*W,C); t2=f2.permute(0,2,3,1).reshape(B,H*W,C)
        n1,n2=self.na(t1),self.nb(t2)
        c1=t1+self.attn12(n1,n2,n2,need_weights=False)[0]
        c2=t2+self.attn21(n2,n1,n1,need_weights=False)[0]
        out=self.proj(torch.cat([self.nc(c1),self.nd(c2)],dim=-1))
        return out.reshape(B,H,W,C).permute(0,3,1,2)

class SiameseSwinRefined(nn.Module):
    def __init__(self, num_classes=8, in_chans=9, img_size=128,
                 backbone_name="swin_tiny_patch4_window7_224",
                 pretrained=True, attn_heads=8, cross_heads=8, num_layers=3):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained,
                            features_only=True, img_size=img_size, in_chans=3)
        widen_patch_embed(self.backbone, in_chans)
        self.chs = self.backbone.feature_info.channels()

        self.self_attn   = BottleneckSelfAttention(self.chs[3], attn_heads, num_layers)
        self.aspp        = ASPP(self.chs[3], self.chs[3], rates=[1,2,4])
        self.cross_attn  = CrossTransformerBlock(self.chs[3], cross_heads)

        self.dec3 = UNetDecoderBlock(self.chs[3], self.chs[2], self.chs[2])
        self.dec2 = UNetDecoderBlock(self.chs[2], self.chs[1], self.chs[1])
        self.dec1 = UNetDecoderBlock(self.chs[1], self.chs[0], self.chs[0])

        self.lulc_head = nn.Sequential(
            nn.Conv2d(self.chs[0], self.chs[0]//2, 3, padding=1),
            nn.BatchNorm2d(self.chs[0]//2), nn.ReLU(True),
            nn.Conv2d(self.chs[0]//2, num_classes, 1))

        self.change_head = nn.Sequential(
            nn.Conv2d(self.chs[3], self.chs[3]//2, 3, padding=1),
            nn.BatchNorm2d(self.chs[3]//2), nn.ReLU(True),
            nn.Conv2d(self.chs[3]//2, 1, 1))

        self.img_size = img_size

    def encode(self, x):
        return [to_nchw(f, self.chs) for f in self.backbone(x)]

    def forward(self, x1, x2):
        f1, f2 = self.encode(x1), self.encode(x2)
        f1a, f2a = self.self_attn(f1[3]), self.self_attn(f2[3])
        f1p, f2p = self.aspp(f1a), self.aspp(f2a)

        s1,s2,s3,_ = f2
        d3 = self.dec3(f2p, s3); d2 = self.dec2(d3,s2); d1 = self.dec1(d2,s1)
        lulc = F.interpolate(self.lulc_head(d1),(self.img_size,self.img_size),mode="bilinear",align_corners=False)

        fused = self.cross_attn(f1p, f2p)
        change = F.interpolate(self.change_head(fused),(self.img_size,self.img_size),mode="bilinear",align_corners=False)
        return lulc, change.squeeze(1)

## Load Checkpoint

In [5]:
model = SiameseSwinRefined(
    num_classes=NUM_CLASSES, in_chans=CONFIG["in_chans"], img_size=CONFIG["img_size"],
    backbone_name=CONFIG["backbone"], pretrained=False,
    attn_heads=CONFIG["attn_heads"], cross_heads=CONFIG["cross_attn_heads"],
    num_layers=CONFIG["num_layers"]
).to(device)

ckpt_path = os.path.join("/kaggle/input/datasets/user/lulc-model-1", CKPT_NAME)
checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")


Loaded checkpoint from epoch 158


## TTA

In [6]:
def tta_predict(model, x1, x2):
    lulc_sum = None
    change_sum = None
    for rotation in range(4):
        rotated_x1 = torch.rot90(x1, rotation, (2, 3))
        rotated_x2 = torch.rot90(x2, rotation, (2, 3))
        with torch.no_grad(), torch.amp.autocast("cuda", enabled=device.type == "cuda"):
            lulc_logits, change_logits = model(rotated_x1, rotated_x2)
        lulc_logits = torch.rot90(lulc_logits, -rotation, (2, 3))
        change_logits = torch.rot90(change_logits.unsqueeze(1), -rotation, (2, 3)).squeeze(1)
        lulc_sum = lulc_logits if lulc_sum is None else lulc_sum + lulc_logits
        change_sum = change_logits if change_sum is None else change_sum + change_logits
    return lulc_sum / 4, change_sum / 4

In [7]:
CLASS_NAMES = ["Trees", "Shrubland", "Grassland", "Cropland", "Built-up", "Bare/Sparse", "Water", "Wetland"]

LULC_COLORS = np.array([
    [0, 100, 0], [139, 90, 43], [124, 252, 0], [255, 215, 0],
    [255, 0, 0], [210, 180, 140], [0, 0, 255], [0, 128, 128],
], dtype=np.float32) / 255.0

def label_to_rgb(label_map, colors=LULC_COLORS):
    rgb = np.zeros((*label_map.shape, 3), dtype=np.float32)
    for class_index in range(len(colors)):
        rgb[label_map == class_index] = colors[class_index]
    return rgb

## City-Wide Inference

In [8]:
YEAR_T1 = 2017
YEAR_T2 = 2025

PIXEL_SIZE_M = 10
PATCH_SIZE = CONFIG["img_size"]
PATCH_AREA_KM2 = (PATCH_SIZE * PIXEL_SIZE_M / 1000) ** 2
PIXEL_AREA_KM2 = (PIXEL_SIZE_M / 1000) ** 2

print(f"Patch area: {PATCH_AREA_KM2:.4f} km²")
print(f"Pixel area: {PIXEL_AREA_KM2:.6f} km²")
print(f"Total patches: {n_patches}")
print(f"Total area: {n_patches * PATCH_AREA_KM2:.2f} km²")

Patch area: 1.6384 km²
Pixel area: 0.000100 km²
Total patches: 3468
Total area: 5681.97 km²


In [9]:
preds_t1 = np.zeros((n_patches, PATCH_SIZE, PATCH_SIZE), dtype=np.uint8)
preds_t2 = np.zeros((n_patches, PATCH_SIZE, PATCH_SIZE), dtype=np.uint8)

BATCH = 16
model.eval()
print(f"Running inference on {n_patches} patches for {YEAR_T1} and {YEAR_T2}...")
start = time.time()

for i in range(0, n_patches, BATCH):
    end_idx = min(i + BATCH, n_patches)
    batch_imgs1 = []
    batch_imgs2 = []
    for j in range(i, end_idx):
        batch_imgs1.append(normalize_bands(to_nine_band(YEAR_ARRAYS[YEAR_T1][j])))
        batch_imgs2.append(normalize_bands(to_nine_band(YEAR_ARRAYS[YEAR_T2][j])))

    t1 = torch.from_numpy(np.stack(batch_imgs1)).to(device)
    t2 = torch.from_numpy(np.stack(batch_imgs2)).to(device)

    lulc1, _ = tta_predict(model, t1, t1)
    lulc2, _ = tta_predict(model, t1, t2)

    preds_t1[i:end_idx] = lulc1.argmax(1).cpu().numpy().astype(np.uint8)
    preds_t2[i:end_idx] = lulc2.argmax(1).cpu().numpy().astype(np.uint8)

    if (i // BATCH) % 50 == 0:
        elapsed = time.time() - start
        print(f"  Processed {end_idx}/{n_patches} patches ({elapsed:.0f}s)")

total_time = time.time() - start
print(f"City-wide inference complete in {total_time:.0f}s ({n_patches / total_time:.1f} patches/sec)")

Running inference on 3468 patches for 2017 and 2025...
  Processed 16/3468 patches (3s)
  Processed 816/3468 patches (21s)
  Processed 1616/3468 patches (40s)
  Processed 2416/3468 patches (59s)
  Processed 3216/3468 patches (78s)
City-wide inference complete in 84s (41.2 patches/sec)


## City-Wide Mosaic

In [10]:
# ── Grid geometry derived from patch stride ──────────────────────────────
# Patches extracted with patch_size=128, stride=64 (50% overlap)
# from original raster ~4454×3341 px:
#   N_ROWS = floor((4454-128)/64)+1 = 68
#   N_COLS = floor((3341-128)/64)+1 = 51
#   Total  = 68 × 51 = 3468  ✓
N_ROWS, N_COLS, STRIDE = 68, 51, 64
CANVAS_H = (N_ROWS - 1) * STRIDE + PATCH_SIZE   # 4416
CANVAS_W = (N_COLS - 1) * STRIDE + PATCH_SIZE   # 3328

# Sliding-window probability accumulation — eliminates patch-boundary seams
# and gives correct per-pixel majority vote when patches overlap.
def build_canvas(preds_array):
    votes = np.zeros((NUM_CLASSES, CANVAS_H, CANVAS_W), dtype=np.float32)
    count = np.zeros((CANVAS_H, CANVAS_W), dtype=np.float32)
    for i in range(N_ROWS * N_COLS):
        r = i // N_COLS
        c = i % N_COLS
        y = r * STRIDE
        x = c * STRIDE
        p = preds_array[i]
        for cls in range(NUM_CLASSES):
            votes[cls, y:y+PATCH_SIZE, x:x+PATCH_SIZE] += (p == cls).astype(np.float32)
        count[y:y+PATCH_SIZE, x:x+PATCH_SIZE] += 1
    return np.argmax(votes, axis=0).astype(np.uint8)

print("Building T1 canvas (sliding-window vote)...")
mosaic_label_t1 = build_canvas(preds_t1)
print("Building T2 canvas...")
mosaic_label_t2 = build_canvas(preds_t2)

mosaic_rgb_t1 = label_to_rgb(mosaic_label_t1)
mosaic_rgb_t2 = label_to_rgb(mosaic_label_t2)

# ── Corrected area quantification ────────────────────────────────────────────
# Each pixel in the canvas represents exactly one 10×10 m Sentinel-2 pixel.
UNIQUE_PIXELS = CANVAS_H * CANVAS_W          # 14,696,448 px
UNIQUE_AREA_KM2 = UNIQUE_PIXELS * PIXEL_AREA_KM2  # ≈ 146.96 km²
print(f"\nCanvas size : {CANVAS_H}×{CANVAS_W} px = {UNIQUE_PIXELS:,} unique pixels")
print(f"Coverage    : {UNIQUE_AREA_KM2:.2f} km²  (≈147 km² of Chennai district)")

# ── Legend patches ───────────────────────────────────────────────────────────
import matplotlib.patches as mpatches
legend_patches = [
    mpatches.Patch(color=LULC_COLORS[c], label=CLASS_NAMES[c])
    for c in range(NUM_CLASSES)
]

fig, axes = plt.subplots(1, 2, figsize=(20, 26))
axes[0].imshow(mosaic_rgb_t1)
axes[0].set_title(f"Chennai LULC — {YEAR_T1}", fontsize=14, fontweight="bold")
axes[0].axis("off")
axes[1].imshow(mosaic_rgb_t2)
axes[1].set_title(f"Chennai LULC — {YEAR_T2}", fontsize=14, fontweight="bold")
axes[1].axis("off")
fig.legend(handles=legend_patches, loc="lower center", ncol=4,
           fontsize=11, frameon=True, title="Land Cover Class",
           title_fontsize=12, bbox_to_anchor=(0.5, 0.01))
fig.suptitle(f"City-Wide LULC Mosaic | Overlap-Averaged Grid 68×51 | ~{UNIQUE_AREA_KM2:.0f} km²",
             fontsize=13, y=0.99)
fig.tight_layout(rect=[0, 0.05, 1, 0.99])
fig.savefig(os.path.join(OUT_DIR, "city_mosaic.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved city_mosaic.png")

Building T1 canvas (sliding-window vote)...
Building T2 canvas...

Canvas size : 4416×3328 px = 14,696,448 unique pixels
Coverage    : 1469.64 km²  (≈147 km² of Chennai district)
Saved city_mosaic.png


## Per-Class Area Quantification (km²)

In [11]:
print(f"\nLand Cover Area Quantification: {YEAR_T1} vs {YEAR_T2}")
print(f"Canvas: {CANVAS_H}x{CANVAS_W} = {CANVAS_H*CANVAS_W*0.0001:.1f} km2 coverage")
print(f"{'Class':>15s} | {'T1 km2':>10s} | {'T2 km2':>10s} | {'Change km2':>12s} | {'Change %':>10s}")
print("-" * 70)

t1_areas = []
t2_areas = []
for c in range(NUM_CLASSES):
    area_t1 = (mosaic_label_t1 == c).sum() * PIXEL_AREA_KM2
    area_t2 = (mosaic_label_t2 == c).sum() * PIXEL_AREA_KM2
    change = area_t2 - area_t1
    pct = (change / area_t1 * 100) if area_t1 > 0 else 0
    t1_areas.append(area_t1)
    t2_areas.append(area_t2)
    print(f"{CLASS_NAMES[c]:>15s} | {area_t1:>10.2f} | {area_t2:>10.2f} | {change:>+12.2f} | {pct:>+9.1f}%")

canvas_total = CANVAS_H * CANVAS_W * PIXEL_AREA_KM2
print(f"{'TOTAL':>15s} | {canvas_total:>10.2f} | {canvas_total:>10.2f} |")


Land Cover Area Quantification: 2017 vs 2025
Canvas: 4416x3328 = 1469.6 km2 coverage
          Class |     T1 km2 |     T2 km2 |   Change km2 |   Change %
----------------------------------------------------------------------
          Trees |     241.85 |     286.62 |       +44.77 |     +18.5%
      Shrubland |      13.19 |      14.47 |        +1.28 |      +9.7%
      Grassland |     128.07 |     105.52 |       -22.54 |     -17.6%
       Cropland |     247.11 |     204.91 |       -42.20 |     -17.1%
       Built-up |     451.92 |     463.84 |       +11.92 |      +2.6%
    Bare/Sparse |      22.87 |      25.08 |        +2.21 |      +9.7%
          Water |     355.89 |     356.18 |        +0.29 |      +0.1%
        Wetland |       8.74 |      13.02 |        +4.28 |     +48.9%
          TOTAL |    1469.64 |    1469.64 |


## Area Comparison Bar Chart

In [12]:
fig, ax = plt.subplots(figsize=(14, 7))
x = np.arange(NUM_CLASSES)
width = 0.35

bars1 = ax.bar(x - width/2, t1_areas, width, label=f"{YEAR_T1}", color="#42A5F5", edgecolor="black", linewidth=0.5)
bars2 = ax.bar(x + width/2, t2_areas, width, label=f"{YEAR_T2}", color="#EF5350", edgecolor="black", linewidth=0.5)

ax.set_ylabel("Area (km²)", fontsize=13)
ax.set_title(f"Chennai LULC Area Comparison: {YEAR_T1} vs {YEAR_T2}", fontsize=15, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right", fontsize=11)
ax.legend(fontsize=12)
ax.grid(axis="y", alpha=0.3)

for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.5, f"{h:.1f}", ha="center", va="bottom", fontsize=8)

fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "city_area_comparison.png"), dpi=200, bbox_inches="tight")
plt.show()
print("Saved city_area_comparison.png")

Saved city_area_comparison.png


## From-To Transition Matrix (km²)

In [13]:
# Compute transition matrix directly from the deduplicated mosaic canvas
# so that area totals match the 1470 km² city canvas exactly (no patch overlap inflation).
t1_flat = mosaic_label_t1.ravel().astype(np.int64)
t2_flat = mosaic_label_t2.ravel().astype(np.int64)
transition_counts = np.bincount(t1_flat * NUM_CLASSES + t2_flat, minlength=NUM_CLASSES * NUM_CLASSES)
transition_matrix = transition_counts.reshape((NUM_CLASSES, NUM_CLASSES)).astype(np.float64)

transition_km2 = transition_matrix * PIXEL_AREA_KM2

try:
    import seaborn as sns
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "-q", "seaborn"])
    import seaborn as sns

fig, ax = plt.subplots(figsize=(12, 10))
mask = transition_km2 < 0.2
sns.heatmap(transition_km2, annot=True, fmt=".1f", cmap="YlOrRd",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax,
            linewidths=0.5, mask=mask)
ax.set_title(f"Land Cover Transition Matrix (km²): {YEAR_T1} → {YEAR_T2}\n(Deduplicated City Canvas: ~1470 km²)", fontsize=13, fontweight="bold")
ax.set_xlabel(f"To ({YEAR_T2})", fontsize=12)
ax.set_ylabel(f"From ({YEAR_T1})", fontsize=12)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "transition_matrix_km2.png"), dpi=200, bbox_inches="tight")
plt.show()
print("Saved transition_matrix_km2.png (canvas-accurate)")

Saved transition_matrix_km2.png (canvas-accurate)
